In [3]:
# ── CELL 1 — Imports ─────────────────────────────────────────
import pyreadstat
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.3f}'.format)
print("✅ Imports done")

✅ Imports done


In [ ]:
# ── CELL 2 — Load ONLY the columns you need ──────────────────
# This avoids loading all 1639 columns into memory
COLS_NEEDED = [
    # Anthropometric
    'HW1','HW2','HW3','HW5','HW6','HW7','HW8',
    'HW56','HW57','HW70','HW71','HW72',
    # Socioeconomic
    'HC61','HC63','V025','V005',
    # Health history
    'H11','H22','H43',
    # Sanitation
    'V116','V113','V469A',
    # Food / dietary recall
    'V414A','V414E','V414F','V414G','V414I',
    'V414J','V414K','V414L','V414M','V414N',
    'V414O','V414P','V414S',
    # State / district
    'SSTATE','SDIST'
]

df, meta = pyreadstat.read_sav(
    'IAKR7EFL.SAV',
    usecols=COLS_NEEDED,
    apply_value_formats=False    # ← CRITICAL: get raw numbers not labels
)
 
print(f"Shape : {df.shape}")
print(f"Cols  : {df.columns.tolist()}")
print("✅ Loaded")

NameError: name 'pyreadstat' is not defined

In [6]:
# HW5 (WHZ) — stored as integer × 100, missing code = 9998
df['HW5'] = pd.to_numeric(df['HW5'], errors='coerce')
df['HW5'] = df['HW5'].replace([9998, 9980, 9996, 9997, 9999], np.nan)
df['HW5'] = df['HW5'] / 100
df.loc[df['HW5'].abs() > 6, 'HW5'] = np.nan

# HW70 (HAZ) — same encoding
df['HW70'] = pd.to_numeric(df['HW70'], errors='coerce')
df['HW70'] = df['HW70'].replace([9998, 9980, 9996, 9997, 9999], np.nan)
df['HW70'] = df['HW70'] / 100
df.loc[df['HW70'].abs() > 6, 'HW70'] = np.nan

print(f"HW5  → min={df['HW5'].min():.2f}  max={df['HW5'].max():.2f}  "
      f"mean={df['HW5'].mean():.2f}  missing={df['HW5'].isna().sum():,}")
print(f"HW70 → min={df['HW70'].min():.2f}  max={df['HW70'].max():.2f}  "
      f"mean={df['HW70'].mean():.2f}  missing={df['HW70'].isna().sum():,}")

HW5  → min=-6.00  max=6.00  mean=-1.21  missing=36,050
HW70 → min=-6.00  max=6.00  mean=-1.31  missing=26,895


In [7]:
# HW3 (weight) — raw median=863, so stored as grams (÷1000 to get kg)
# OR stored as kg × 100 — let's check: 863/100 = 8.63kg (plausible for ~2yr old ✅)
df['HW3'] = pd.to_numeric(df['HW3'], errors='coerce')
df['HW3'] = df['HW3'].replace([9996, 9997, 9998, 9999], np.nan)
df['HW3'] = df['HW3'] / 100
df.loc[(df['HW3'] < 1.5) | (df['HW3'] > 30), 'HW3'] = np.nan

# HW8 (height) — raw sample has negatives [-598] and large [9998]
# negative values = another z-score style encoding, missing=9998
# actual height sample [122, 149, 335, 294] → 335/10=33.5cm (too low)
# 335 could be mm → 335mm = 33.5cm (too low) OR already cm?
# median=-146 suggests it's ALSO stored as integer×100 like z-scores
# -146/100 = -1.46 (that's a z-score, not height!)
# So HW8 in your file is actually height-for-age z-score duplicate or mislabeled
# Let's inspect more carefully first
print(f"HW3 → min={df['HW3'].min():.2f}  max={df['HW3'].max():.2f}  "
      f"median={df['HW3'].median():.2f}  missing={df['HW3'].isna().sum():,}")

print(f"\nHW8 raw describe:")
print(df['HW8'].describe())
print(f"\nHW8 non-missing sample (first 15):")
print(df['HW8'].dropna().head(15).tolist())
print(f"\nHW8 positive values only (first 15):")
print(df['HW8'][df['HW8'] > 0].head(15).tolist())
print(f"\nHW8 value counts (top 15):")
print(df['HW8'].value_counts().head(15))

HW3 → min=2.00  max=14.00  median=8.57  missing=21,756

HW8 raw describe:
count   210989.000
mean       533.786
std       2537.308
min       -598.000
25%       -222.000
50%       -146.000
75%        -50.000
max       9998.000
Name: HW8, dtype: float64

HW8 non-missing sample (first 15):
[9998.0, 122.0, 149.0, 9998.0, 9998.0, 335.0, 9998.0, 294.0, 9998.0, -218.0, 9998.0, -192.0, -201.0, -257.0, -195.0]

HW8 positive values only (first 15):
[9998.0, 122.0, 149.0, 9998.0, 9998.0, 335.0, 9998.0, 294.0, 9998.0, 9998.0, 9998.0, 9998.0, 16.0, 45.0, 9998.0]

HW8 value counts (top 15):
HW8
9998.000    14119
-179.000      756
-183.000      753
-186.000      738
-208.000      733
-191.000      731
-188.000      730
-193.000      730
-187.000      730
-194.000      729
-153.000      727
-173.000      727
-200.000      724
-169.000      724
-156.000      720
Name: count, dtype: int64


In [8]:
# HW8 — contains z-score data (integer × 100), NOT height in cm
# median=-146 → -1.46 HAZ, sample [-179,-183,-186] → all valid HAZ values
# Positive outliers [122,149,335,294] → /100 = [1.22,1.49,3.35,2.94] — valid z-scores
# We already have HW70 as HAZ so we rename HW8 as HAZ_alt for cross-check

df['HW8'] = pd.to_numeric(df['HW8'], errors='coerce')
df['HW8'] = df['HW8'].replace([9998, 9980, 9996, 9997, 9999], np.nan)
df['HW8'] = df['HW8'] / 100
df.loc[df['HW8'].abs() > 6, 'HW8'] = np.nan

# Rename to reflect what it actually is
df.rename(columns={'HW8': 'HAZ_alt'}, inplace=True)

print(f"HW8 (HAZ_alt) → min={df['HAZ_alt'].min():.2f}  "
      f"max={df['HAZ_alt'].max():.2f}  "
      f"mean={df['HAZ_alt'].mean():.2f}  "
      f"missing={df['HAZ_alt'].isna().sum():,}")

# HW3 already fixed (2.0–14.0 kg is correct for under-5) ✅
print(f"HW3 (weight kg) → min={df['HW3'].min():.2f}  "
      f"max={df['HW3'].max():.2f}  "
      f"median={df['HW3'].median():.2f}  "
      f"missing={df['HW3'].isna().sum():,}")

# HW56 (haemoglobin) — raw median=103, stored as g/dL × 10
df['HW56'] = pd.to_numeric(df['HW56'], errors='coerce')
df['HW56'] = df['HW56'].replace([999, 9998, 9999], np.nan)
df['HW56'] = df['HW56'] / 10
df.loc[(df['HW56'] < 3) | (df['HW56'] > 20), 'HW56'] = np.nan
print(f"HW56 (Hb g/dL) → min={df['HW56'].min():.1f}  "
      f"max={df['HW56'].max():.1f}  "
      f"mean={df['HW56'].mean():.1f}  "
      f"missing={df['HW56'].isna().sum():,}")

# HW57 (anaemia classification)
df['HW57'] = pd.to_numeric(df['HW57'], errors='coerce')
df['HW57'] = df['HW57'].replace([8, 9], np.nan)
print(f"\nHW57 distribution:")
print(df['HW57'].value_counts(dropna=False).sort_index())

HW8 (HAZ_alt) → min=-5.98  max=5.98  mean=-1.45  missing=36,050
HW3 (weight kg) → min=2.00  max=14.00  median=8.57  missing=21,756
HW56 (Hb g/dL) → min=3.0  max=20.0  mean=10.3  missing=49,301

HW57 distribution:
HW57
1.000     4050
2.000    66709
3.000    52703
4.000    60393
NaN      49065
Name: count, dtype: int64


In [9]:
# Remap HW57 to standard 0,1,2,3 scale
df['HW57'] = df['HW57'].map({1: 0, 2: 1, 3: 2, 4: 3})
# 0=not anaemic, 1=mild, 2=moderate, 3=severe

print("HW57 remapped:")
print(df['HW57'].value_counts(dropna=False).sort_index())

# Binary health columns — replace don't know / missing codes
for col in ['H11', 'H22', 'H43']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].replace([8, 9], np.nan)

print(f"\nH11 (fever)      : {df['H11'].value_counts(dropna=False).to_dict()}")
print(f"H22 (Vit A supp) : {df['H22'].value_counts(dropna=False).to_dict()}")
print(f"H43 (deworming)  : {df['H43'].value_counts(dropna=False).to_dict()}")

# Food columns — replace don't know / missing codes
food_cols = [c for c in df.columns if c.startswith('V414')]
for col in food_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace([8, 9], np.nan)

print(f"\nFood columns     : {food_cols}")
print(f"Food NaN total   : {df[food_cols].isnull().sum().sum():,}")

# Age-aware fill
df['HW1'] = pd.to_numeric(df['HW1'], errors='coerce')

# Food recall not asked for under-6 months → fill 0
for col in food_cols:
    df.loc[df['HW1'] < 6, col] = df.loc[df['HW1'] < 6, col].fillna(0)
    df[col] = df[col].fillna(0)

# Deworming not given under 12 months → fill 0
if 'H43' in df.columns:
    df.loc[df['HW1'] < 12, 'H43'] = df.loc[df['HW1'] < 12, 'H43'].fillna(0)
    df['H43'] = df['H43'].fillna(0)

print(f"\nAfter age-aware fill:")
print(f"Food NaN total   : {df[food_cols].isnull().sum().sum()}")
print(f"H43 NaN total    : {df['H43'].isna().sum():,}")

HW57 remapped:
HW57
0.000     4050
1.000    66709
2.000    52703
3.000    60393
NaN      49065
Name: count, dtype: int64

H11 (fever)      : {0.0: 208451, 2.0: 15334, nan: 9135}
H22 (Vit A supp) : {0.0: 196178, 1.0: 27700, nan: 9042}
H43 (deworming)  : {nan: 101536, 0.0: 81118, 1.0: 50266}

Food columns     : ['V414A', 'V414E', 'V414F', 'V414G', 'V414I', 'V414J', 'V414K', 'V414L', 'V414M', 'V414N', 'V414O', 'V414P', 'V414S']
Food NaN total   : 1,369,158

After age-aware fill:
Food NaN total   : 0
H43 NaN total    : 0


In [10]:
# H11 has 0=No, 2=Yes → remap to standard 0=No, 1=Yes
df['H11'] = df['H11'].map({0: 0, 2: 1})
print("H11 remapped:")
print(df['H11'].value_counts(dropna=False).to_dict())

# H22 check — 0=No, 1=Yes ✅ already correct
# H43 check — 0=No, 1=Yes ✅ already correct

# Add age flag columns (useful features)
df['age_under_6']  = (df['HW1'] < 6).astype(int)
df['age_under_12'] = (df['HW1'] < 12).astype(int)
df['age_under_24'] = (df['HW1'] < 24).astype(int)

# ── Impute remaining missing values ──────────────────────────

# Continuous → median
for col in ['HW3', 'HW5', 'HW70', 'HAZ_alt', 'HW56']:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"{col} median imputed with {median_val:.3f}")

# Categorical → mode
for col in ['HW1','HW2','HW57','HW6','HW7','HW71','HW72',
            'HC61','HC63','V025','H11','H22','V116']:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"{col} mode imputed with {mode_val}")

print(f"\nTotal remaining NaN : {df.isnull().sum().sum()}")
print(f"Shape               : {df.shape}")

H11 remapped:
{0.0: 208451, 1.0: 15334, nan: 9135}
HW3 median imputed with 8.570
HW5 median imputed with -1.280
HW70 median imputed with -1.450
HAZ_alt median imputed with -1.560
HW56 median imputed with 10.300
HW1 mode imputed with 53.0
HW2 mode imputed with 9994.0
HW57 mode imputed with 1.0
HW6 mode imputed with 99998.0
HW7 mode imputed with 9998.0
HW71 mode imputed with 9998.0
HW72 mode imputed with 9998.0
V025 mode imputed with 2.0
H11 mode imputed with 0.0
H22 mode imputed with 0.0
V116 mode imputed with 12.0

Total remaining NaN : 0
Shape               : (232920, 36)


In [11]:
# ── Clean flag columns (binary 0/1) ──────────────────────────
# These are derived flags: 0=No, 1=Yes, everything else = missing

flag_cols = ['HW2', 'HW6', 'HW7', 'HW71', 'HW72']

for col in flag_cols:
    if col in df.columns:
        # Check raw distribution first
        print(f"\n{col} before fix:")
        print(df[col].value_counts(dropna=False).sort_index().head(8))

        df[col] = pd.to_numeric(df[col], errors='coerce')

        # Keep only valid values, everything else → NaN
        if col == 'HW2':
            # Sex: 1=male, 2=female only
            df[col] = df[col].where(df[col].isin([1, 2]), other=np.nan)
        else:
            # Flags: 0=No, 1=Yes only
            df[col] = df[col].where(df[col].isin([0, 1]), other=np.nan)

        print(f"{col} after fix:")
        print(df[col].value_counts(dropna=False).sort_index())


HW2 before fix:
HW2
5.000     12
6.000      1
7.000      2
9.000      2
10.000     3
11.000     1
12.000     1
13.000     8
Name: count, dtype: int64
HW2 after fix:
HW2
NaN    232920
Name: count, dtype: int64

HW6 before fix:
HW6
7501.000    1
7503.000    1
7509.000    2
7511.000    1
7515.000    1
7520.000    1
7522.000    2
7524.000    1
Name: count, dtype: int64
HW6 after fix:
HW6
NaN    232920
Name: count, dtype: int64

HW7 before fix:
HW7
0.000    2169
1.000    2154
2.000    1500
3.000    1292
4.000    1096
5.000     996
6.000     963
7.000     913
Name: count, dtype: int64
HW7 after fix:
HW7
0.000      2169
1.000      2154
NaN      228597
Name: count, dtype: int64

HW71 before fix:
HW71
-600.000     3
-599.000    11
-598.000     7
-597.000     8
-596.000    13
-595.000     7
-594.000     6
-593.000    10
Name: count, dtype: int64
HW71 after fix:
HW71
0.000       321
1.000       380
NaN      232219
Name: count, dtype: int64

HW72 before fix:
HW72
-500.000    17
-499.000    38
-49

In [13]:
# ── Clean flag columns ────────────────────────────────────────
flag_cols = ['HW2', 'HW6', 'HW7', 'HW71', 'HW72']

for col in flag_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if col == 'HW2':
            df[col] = df[col].where(df[col].isin([1, 2]), other=np.nan)
        else:
            df[col] = df[col].where(df[col].isin([0, 1]), other=np.nan)

# ── Clean HC61 and HC63 ───────────────────────────────────────
for col in ['HC61', 'HC63']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Safe re-impute ────────────────────────────────────────────
def safe_mode_impute(df, col):
    """Impute with mode — if column all NaN, fill with 0"""
    if col not in df.columns:
        return
    if df[col].isna().all():
        print(f"{col} → ALL NaN — filling with 0")
        df[col] = df[col].fillna(0)
        return
    mode_val = df[col].mode()
    if len(mode_val) == 0:
        print(f"{col} → empty mode — filling with 0")
        df[col] = df[col].fillna(0)
        return
    df[col] = df[col].fillna(mode_val.iloc[0])
    print(f"{col} → mode={mode_val.iloc[0]}  "
          f"NaN remaining={df[col].isna().sum()}")

cols_to_impute = ['HW2','HW6','HW7','HW71','HW72',
                  'HC61','HC63','V025','H11','H22','V116']

for col in cols_to_impute:
    safe_mode_impute(df, col)

print(f"\nTotal remaining NaN : {df.isnull().sum().sum()}")
print(f"Shape               : {df.shape}")

HW2 → ALL NaN — filling with 0
HW6 → ALL NaN — filling with 0
HW7 → mode=0.0  NaN remaining=0
HW71 → mode=1.0  NaN remaining=0
HW72 → mode=1.0  NaN remaining=0
V025 → mode=2.0  NaN remaining=0
H11 → mode=0.0  NaN remaining=0
H22 → mode=0.0  NaN remaining=0
V116 → mode=12.0  NaN remaining=0

Total remaining NaN : 0
Shape               : (232920, 36)


In [14]:
# ── Check what columns we actually have ──────────────────────
print("Current columns:")
print(df.columns.tolist())
print(f"\nTotal columns: {len(df.columns)}")

# ── Check HW2 and HW6 ────────────────────────────────────────
print(f"\nHW2 unique values: {df['HW2'].unique()}")
print(f"HW2 value counts:\n{df['HW2'].value_counts(dropna=False)}")

print(f"\nHW6 unique values: {df['HW6'].unique()}")
print(f"HW6 value counts:\n{df['HW6'].value_counts(dropna=False)}")

# ── Check if HC61, HC63 were loaded ──────────────────────────
for col in ['HC61', 'HC63', 'SSTATE']:
    if col in df.columns:
        print(f"\n{col} is present:")
        print(df[col].value_counts(dropna=False).head(6))
    else:
        print(f"\n{col} → NOT in dataframe")
        

Current columns:
['V005', 'V025', 'V113', 'V116', 'V414A', 'V414E', 'V414F', 'V414G', 'V414I', 'V414J', 'V414K', 'V414L', 'V414M', 'V414N', 'V414O', 'V414P', 'V414S', 'H11', 'H22', 'H43', 'HW1', 'HW2', 'HW3', 'HW5', 'HW6', 'HW7', 'HAZ_alt', 'HW56', 'HW57', 'HW70', 'HW71', 'HW72', 'SDIST', 'age_under_6', 'age_under_12', 'age_under_24']

Total columns: 36

HW2 unique values: [0.]
HW2 value counts:
HW2
0.000    232920
Name: count, dtype: int64

HW6 unique values: [0.]
HW6 value counts:
HW6
0.000    232920
Name: count, dtype: int64

HC61 → NOT in dataframe

HC63 → NOT in dataframe

SSTATE → NOT in dataframe


In [15]:
# ── Reload with the missing columns added ────────────────────
COLS_NEEDED = [
    'V005', 'V025', 'V113', 'V116',
    'V414A', 'V414E', 'V414F', 'V414G', 'V414I',
    'V414J', 'V414K', 'V414L', 'V414M', 'V414N',
    'V414O', 'V414P', 'V414S',
    'H11', 'H22', 'H43',
    'HW1', 'HW2', 'HW3', 'HW5', 'HW6', 'HW7',
    'HW8', 'HW56', 'HW57', 'HW70', 'HW71', 'HW72',
    'HC61', 'HC63',        # wealth index + mother education
    'SSTATE', 'SDIST',     # state + district
    'V469A'                # cooking fuel
]

df, meta = pyreadstat.read_sav(
    'IAKR7EFL.SAV',
    usecols=COLS_NEEDED,
    apply_value_formats=False
)

print(f"Shape  : {df.shape}")
print(f"Loaded columns:\n{df.columns.tolist()}")

# ── Check which ones actually loaded ─────────────────────────
wanted   = set(COLS_NEEDED)
got      = set(df.columns.tolist())
missing  = wanted - got

print(f"\nRequested : {len(wanted)}")
print(f"Loaded    : {len(got)}")
print(f"Not found : {missing if missing else 'None — all loaded ✅'}")

Shape  : (232920, 33)
Loaded columns:
['V005', 'V025', 'V113', 'V116', 'V414A', 'V414E', 'V414F', 'V414G', 'V414I', 'V414J', 'V414K', 'V414L', 'V414M', 'V414N', 'V414O', 'V414P', 'V414S', 'H11', 'H22', 'H43', 'HW1', 'HW2', 'HW3', 'HW5', 'HW6', 'HW7', 'HW8', 'HW56', 'HW57', 'HW70', 'HW71', 'HW72', 'SDIST']

Requested : 37
Loaded    : 33
Not found : {'V469A', 'SSTATE', 'HC63', 'HC61'}


In [16]:
# Load ALL column names from the file without loading data
_, meta_full = pyreadstat.read_sav(
    'IAKR7EFL.SAV',
    apply_value_formats=False,
    row_limit=1          # load just 1 row to get all column names fast
)

all_cols = meta_full.column_names
print(f"Total columns in file: {len(all_cols)}")

# ── Search for wealth index ───────────────────────────────────
wealth_matches = [c for c in all_cols if 'wealth' in c.lower() 
                  or 'wlth' in c.lower()
                  or c.upper().startswith('HC6')]
print(f"\nWealth-related columns: {wealth_matches}")

# ── Search for mother education ───────────────────────────────
edu_matches = [c for c in all_cols if 'edu' in c.lower()
               or c.upper().startswith('HC6')]
print(f"Education-related columns: {edu_matches}")

# ── Search for state ──────────────────────────────────────────
state_matches = [c for c in all_cols if 'state' in c.lower()
                 or c.upper().startswith('SSTATE')
                 or c.upper().startswith('V024')
                 or c.upper() in ['SSTATE','V101','V024']]
print(f"State-related columns: {state_matches}")

# ── Search for cooking fuel ───────────────────────────────────
fuel_matches = [c for c in all_cols if 'fuel' in c.lower()
                or c.upper().startswith('V469')]
print(f"Fuel-related columns: {fuel_matches}")

# ── Also print all HC columns ─────────────────────────────────
hc_cols = [c for c in all_cols if c.upper().startswith('HC')]
print(f"\nAll HC columns ({len(hc_cols)}): {hc_cols}")

# ── All V0xx columns that might be socioeconomic ─────────────
v0_cols = [c for c in all_cols if c.upper().startswith('V0')]
print(f"\nAll V0xx columns ({len(v0_cols)}): {v0_cols}")

Total columns in file: 1644

Wealth-related columns: []
Education-related columns: []
State-related columns: ['V024', 'V101']
Fuel-related columns: ['V469E', 'V469F', 'V469X']

All HC columns (0): []

All V0xx columns (43): ['V000', 'V001', 'V002', 'V003', 'V004', 'V005', 'V006', 'V007', 'V008', 'V008A', 'V009', 'V010', 'V011', 'V012', 'V013', 'V014', 'V015', 'V016', 'V017', 'V018', 'V019', 'V019A', 'V020', 'V021', 'V022', 'V023', 'V024', 'V025', 'V026', 'V027', 'V028', 'V029', 'V030', 'V031', 'V032', 'V034', 'V040', 'V042', 'V044', 'V045A', 'V045B', 'V045C', 'V046']


In [17]:
# Search more broadly across all 1644 columns
all_cols = meta_full.column_names

# Print ALL column names so we can scan them
print("ALL COLUMNS IN FILE:")
for i, col in enumerate(all_cols):
    print(f"{i:4d}  {col}")

ALL COLUMNS IN FILE:
   0  CASEID
   1  MIDX
   2  V000
   3  V001
   4  V002
   5  V003
   6  V004
   7  V005
   8  V006
   9  V007
  10  V008
  11  V008A
  12  V009
  13  V010
  14  V011
  15  V012
  16  V013
  17  V014
  18  V015
  19  V016
  20  V017
  21  V018
  22  V019
  23  V019A
  24  V020
  25  V021
  26  V022
  27  V023
  28  V024
  29  V025
  30  V026
  31  V027
  32  V028
  33  V029
  34  V030
  35  V031
  36  V032
  37  V034
  38  V040
  39  V042
  40  V044
  41  V045A
  42  V045B
  43  V045C
  44  V046
  45  V101
  46  V102
  47  V103
  48  V104
  49  V105
  50  V105A
  51  V106
  52  V107
  53  V113
  54  V115
  55  V116
  56  V119
  57  V120
  58  V121
  59  V122
  60  V123
  61  V124
  62  V125
  63  V127
  64  V128
  65  V129
  66  V130
  67  V131
  68  V133
  69  V134
  70  V135
  71  V136
  72  V137
  73  V138
  74  V139
  75  V140
  76  V141
  77  V149
  78  V150
  79  V151
  80  V152
  81  V153
  82  AWFACTT
  83  AWFACTU
  84  AWFACTR
  85  AWFACTE
  86  AWFACTW

In [18]:
# Targeted search across all columns
keywords = ['wealth', 'educ', 'mother', 'hh', 
            'v106', 'v107', 'v130', 'v131',
            'v190', 'v191', 'v135', 'v136']

print("Keyword matches:")
for col in all_cols:
    for kw in keywords:
        if kw.lower() in col.lower():
            print(f"  {col}")
            break

# Print V1xx columns
v1_cols = [c for c in all_cols if c.upper().startswith('V1')]
print(f"\nAll V1xx columns ({len(v1_cols)}):")
print(v1_cols)

# Print V190+ columns (wealth index is usually V190 in DHS)
v19_cols = [c for c in all_cols if c.upper().startswith('V19')]
print(f"\nV19x columns: {v19_cols}")

# Print columns 200-300 in the list
print(f"\nColumns 200-300:")
print(all_cols[200:300])

Keyword matches:
  V106
  V107
  V130
  V131
  V135
  V136
  V190
  V191
  V190A
  V191A

All V1xx columns (56):
['V101', 'V102', 'V103', 'V104', 'V105', 'V105A', 'V106', 'V107', 'V113', 'V115', 'V116', 'V119', 'V120', 'V121', 'V122', 'V123', 'V124', 'V125', 'V127', 'V128', 'V129', 'V130', 'V131', 'V133', 'V134', 'V135', 'V136', 'V137', 'V138', 'V139', 'V140', 'V141', 'V149', 'V150', 'V151', 'V152', 'V153', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V166', 'V167', 'V168', 'V169A', 'V169B', 'V170', 'V171A', 'V171B', 'V190', 'V191', 'V190A', 'V191A']

V19x columns: ['V190', 'V191', 'V190A', 'V191A']

Columns 200-300:
['V3A00K', 'V3A00L', 'V3A00M', 'V3A00N', 'V3A00O', 'V3A00P', 'V3A00Q', 'V3A00R', 'V3A00S', 'V3A00T', 'V3A00U', 'V3A00V', 'V3A00W', 'V3A00X', 'V3A00Y', 'V3A00Z', 'V3A01', 'V3A02', 'V3A03', 'V3A04', 'V3A05', 'V3A06', 'V3A07', 'V3A08A', 'V3A08B', 'V3A08C', 'V3A08D', 'V3A08E', 'V3A08F', 'V3A08G', 'V3A08H', 'V3A08I', 'V3A08J', 'V3A08K', 'V3A08L', 'V3A08M', 'V3A08N',

In [19]:
COLS_FINAL = [
    # Identifiers
    'V001', 'V002', 'V003',
    # Survey weight
    'V005',
    # Geographic
    'V024', 'V025', 'SDIST',
    # Socioeconomic
    'V106',   # mother education  (replaces HC61)
    'V190',   # wealth index      (replaces HC63)
    # Household
    'V113',   # water source
    'V116',   # toilet type
    'V469E',  # cooking fuel (main)
    # Child health
    'H11', 'H22', 'H43',
    # Anthropometric
    'HW1', 'HW2', 'HW3',
    'HW5', 'HW6', 'HW7',
    'HW8', 'HW56', 'HW57',
    'HW70', 'HW71', 'HW72',
    # Dietary recall
    'V414A', 'V414E', 'V414F', 'V414G', 'V414I',
    'V414J', 'V414K', 'V414L', 'V414M', 'V414N',
    'V414O', 'V414P', 'V414S',
]

df, meta = pyreadstat.read_sav(
    'IAKR7EFL.SAV',
    usecols=COLS_FINAL,
    apply_value_formats=False
)

print(f"Shape   : {df.shape}")

# Confirm all loaded
got     = set(df.columns.tolist())
wanted  = set(COLS_FINAL)
missing = wanted - got
print(f"Missing : {missing if missing else 'None ✅'}")

# Quick check on the new columns
print(f"\nV106 (mother edu) distribution:")
print(df['V106'].value_counts(dropna=False).sort_index().head(8))

print(f"\nV190 (wealth index) distribution:")
print(df['V190'].value_counts(dropna=False).sort_index().head(8))

print(f"\nV024 (state) unique values: {df['V024'].nunique()}")
print(df['V024'].value_counts().head(5))

print(f"\nHW2 (sex) distribution:")
print(df['HW2'].value_counts(dropna=False).sort_index())

Shape   : (232920, 40)
Missing : None ✅

V106 (mother edu) distribution:
V106
0.000     51210
1.000     30081
2.000    119864
3.000     31765
Name: count, dtype: int64

V190 (wealth index) distribution:
V190
1.000    63406
2.000    54463
3.000    45083
4.000    39094
5.000    30874
Name: count, dtype: int64

V024 (state) unique values: 36
V024
9.000     35766
10.000    21040
23.000    16280
8.000     14643
18.000    10645
Name: count, dtype: int64

HW2 (sex) distribution:
HW2
5.000          12
6.000           1
7.000           2
9.000           2
10.000          3
            ...  
778.000         1
9994.000     3333
9995.000     2703
9996.000      178
NaN         14881
Name: count, Length: 341, dtype: int64


In [21]:
# Search for sex/gender column across all 1644 columns
all_cols = meta_full.column_names

# Common DHS names for child sex
sex_candidates = ['B4', 'B4_01', 'BORD', 'V104', 'B15']
print("Checking common DHS sex column names:")
for col in sex_candidates:
    if col in all_cols:
        print(f"  {col} → EXISTS in file")
    else:
        print(f"  {col} → not found")

# Search by keyword
sex_matches = [c for c in all_cols 
               if 'sex' in c.lower() 
               or 'gender' in c.lower()
               or c.upper() in ['B4','B4_01','BIDX']]
print(f"\nKeyword matches: {sex_matches}")

# Print all B-series columns (birth history variables)
b_cols = [c for c in all_cols if c.upper().startswith('B')]
print(f"\nAll B-series columns ({len(b_cols)}):")
print(b_cols)

Checking common DHS sex column names:
  B4 → EXISTS in file
  B4_01 → not found
  BORD → EXISTS in file
  V104 → EXISTS in file
  B15 → EXISTS in file

Keyword matches: ['BIDX', 'B4']

All B-series columns (22):
['BIDX', 'BORD', 'B0', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20']


In [22]:
# ── Reload with B4 included ───────────────────────────────────
COLS_FINAL = [
    'V005', 'V024', 'V025', 'V113', 'V116', 'V469E',
    'V106', 'V190',
    'V414A', 'V414E', 'V414F', 'V414G', 'V414I',
    'V414J', 'V414K', 'V414L', 'V414M', 'V414N',
    'V414O', 'V414P', 'V414S',
    'H11', 'H22', 'H43',
    'HW1', 'HW3', 'HW5', 'HW6', 'HW7',
    'HW8', 'HW56', 'HW57', 'HW70', 'HW71', 'HW72',
    'B4',      # child sex (replaces HW2)
    'SDIST',
]

df, meta = pyreadstat.read_sav(
    'IAKR7EFL.SAV',
    usecols=COLS_FINAL,
    apply_value_formats=False
)

print(f"Shape   : {df.shape}")
got     = set(df.columns.tolist())
wanted  = set(COLS_FINAL)
missing = wanted - got
print(f"Missing : {missing if missing else 'None ✅'}")

# Quick check B4
print(f"\nB4 (sex) distribution:")
print(df['B4'].value_counts(dropna=False).sort_index())

Shape   : (232920, 37)
Missing : None ✅

B4 (sex) distribution:
B4
1.000    120665
2.000    112255
Name: count, dtype: int64


In [23]:
# ════════════════════════════════════════════════════════════
# FULL CLEAN PIPELINE
# ════════════════════════════════════════════════════════════

# ── 1. Sex (B4) ───────────────────────────────────────────────
df['B4'] = pd.to_numeric(df['B4'], errors='coerce')
df['B4'] = df['B4'].where(df['B4'].isin([1, 2]), other=np.nan)
df['B4'] = df['B4'].fillna(df['B4'].dropna().mode().iloc[0])
df['sex'] = df['B4'].map({1: 0, 2: 1})   # 0=male, 1=female
print(f"sex  : {df['sex'].value_counts().to_dict()}")

# ── 2. Z-scores (integer × 100, missing = 9998) ───────────────
for col in ['HW5', 'HW70', 'HW8']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace([9998, 9980, 9996, 9997, 9999], np.nan)
    df[col] = df[col] / 100
    df.loc[df[col].abs() > 6, col] = np.nan

df.rename(columns={'HW8': 'HAZ_alt'}, inplace=True)
print(f"WHZ  : mean={df['HW5'].mean():.2f}  missing={df['HW5'].isna().sum():,}")
print(f"HAZ  : mean={df['HW70'].mean():.2f}  missing={df['HW70'].isna().sum():,}")

# ── 3. Weight (÷100 → kg) ────────────────────────────────────
df['HW3'] = pd.to_numeric(df['HW3'], errors='coerce')
df['HW3'] = df['HW3'].replace([9996, 9997, 9998, 9999], np.nan)
if df['HW3'].dropna().median() > 100:
    df['HW3'] = df['HW3'] / 100
df.loc[(df['HW3'] < 1.5) | (df['HW3'] > 30), 'HW3'] = np.nan
print(f"HW3  : min={df['HW3'].min():.1f}  max={df['HW3'].max():.1f}  "
      f"missing={df['HW3'].isna().sum():,}")

# ── 4. Haemoglobin (÷10 → g/dL) ──────────────────────────────
df['HW56'] = pd.to_numeric(df['HW56'], errors='coerce')
df['HW56'] = df['HW56'].replace([999, 9998, 9999], np.nan)
if df['HW56'].dropna().median() > 50:
    df['HW56'] = df['HW56'] / 10
df.loc[(df['HW56'] < 3) | (df['HW56'] > 20), 'HW56'] = np.nan
print(f"HW56 : min={df['HW56'].min():.1f}  max={df['HW56'].max():.1f}  "
      f"missing={df['HW56'].isna().sum():,}")

# ── 5. Anaemia remap 1-4 → 0-3 ───────────────────────────────
df['HW57'] = pd.to_numeric(df['HW57'], errors='coerce')
df['HW57'] = df['HW57'].replace([8, 9], np.nan)
df['HW57'] = df['HW57'].map({1: 0, 2: 1, 3: 2, 4: 3})
print(f"HW57 : {df['HW57'].value_counts().sort_index().to_dict()}")

# ── 6. Binary flag columns ────────────────────────────────────
for col in ['HW6', 'HW7', 'HW71', 'HW72']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].where(df[col].isin([0, 1]), other=np.nan)

# Re-derive from z-scores if lost
if df['HW6'].isna().all() or df['HW6'].sum() == 0:
    df['HW6'] = (df['HW5'] < -2).astype(float)
    print("HW6 re-derived from HW5")
if df['HW7'].isna().all() or df['HW7'].sum() == 0:
    df['HW7'] = (df['HW5'] < -3).astype(float)
    print("HW7 re-derived from HW5")
if df['HW71'].isna().all() or df['HW71'].sum() == 0:
    df['HW71'] = (df['HW70'] < -2).astype(float)
    print("HW71 re-derived from HW70")
if df['HW72'].isna().all() or df['HW72'].sum() == 0:
    df['HW72'] = (df['HW5'] < -2).astype(float)
    print("HW72 re-derived from HW5")

print(f"HW6  (wasted)  : 0={int((df['HW6']==0).sum())}  "
      f"1={int((df['HW6']==1).sum())}")
print(f"HW7  (SAM)     : 0={int((df['HW7']==0).sum())}  "
      f"1={int((df['HW7']==1).sum())}")
print(f"HW71 (stunted) : 0={int((df['HW71']==0).sum())}  "
      f"1={int((df['HW71']==1).sum())}")
print(f"HW72 (underweight) : 0={int((df['HW72']==0).sum())}  "
      f"1={int((df['HW72']==1).sum())}")

# ── 7. H11 remap (0=No, 2=Yes → 0/1) ────────────────────────
df['H11'] = pd.to_numeric(df['H11'], errors='coerce')
df['H11'] = df['H11'].replace([8, 9], np.nan)
df['H11'] = df['H11'].map({0: 0, 2: 1})

for col in ['H22', 'H43']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace([8, 9], np.nan)

# ── 8. Food columns + age-aware fill ─────────────────────────
food_cols = [c for c in df.columns if c.startswith('V414')]
for col in food_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace([8, 9], np.nan)

df['HW1'] = pd.to_numeric(df['HW1'], errors='coerce')

for col in food_cols:
    df.loc[df['HW1'] < 6, col] = df.loc[df['HW1'] < 6, col].fillna(0)
    df[col] = df[col].fillna(0)

df.loc[df['HW1'] < 12, 'H43'] = df.loc[df['HW1'] < 12, 'H43'].fillna(0)
df['H43'] = df['H43'].fillna(0)

# ── 9. Age flag columns ───────────────────────────────────────
df['age_under_6']  = (df['HW1'] < 6).astype(int)
df['age_under_12'] = (df['HW1'] < 12).astype(int)
df['age_under_24'] = (df['HW1'] < 24).astype(int)

# ── 10. Impute remaining NaN ──────────────────────────────────
# Continuous → median
for col in ['HW1', 'HW3', 'HW5', 'HW70', 'HAZ_alt', 'HW56']:
    df[col] = df[col].fillna(df[col].median())

# Categorical → safe mode
def safe_fill_mode(df, col):
    if col not in df.columns:
        return
    valid = df[col].dropna()
    if len(valid) == 0:
        df[col] = df[col].fillna(0)
        return
    df[col] = df[col].fillna(valid.mode().iloc[0])

for col in ['sex', 'HW6', 'HW7', 'HW57', 'HW71', 'HW72',
            'V025', 'V106', 'V190', 'V024',
            'H11', 'H22', 'V116', 'V469E']:
    safe_fill_mode(df, col)

print(f"\nTotal remaining NaN : {df.isnull().sum().sum()}")

# ── 11. Final quality report ──────────────────────────────────
print("\n" + "=" * 55)
print("FINAL DATASET QUALITY REPORT")
print("=" * 55)
print(f"Total rows           : {len(df):,}")
print(f"Total columns        : {len(df.columns)}")
print(f"Remaining NaN        : {df.isnull().sum().sum()}")
print()
print(f"WHZ  (HW5)           : {df['HW5'].min():.2f} to "
      f"{df['HW5'].max():.2f}  mean={df['HW5'].mean():.2f}")
print(f"HAZ  (HW70)          : {df['HW70'].min():.2f} to "
      f"{df['HW70'].max():.2f}  mean={df['HW70'].mean():.2f}")
print(f"Weight kg  (HW3)     : {df['HW3'].min():.1f} to "
      f"{df['HW3'].max():.1f}")
print(f"Haemoglobin (HW56)   : {df['HW56'].min():.1f} to "
      f"{df['HW56'].max():.1f}")
print()
print(f"Wasted  (WHZ < -2)   : {(df['HW5'] < -2).sum():,}  "
      f"({(df['HW5'] < -2).mean()*100:.1f}%)")
print(f"SAM     (WHZ < -3)   : {(df['HW5'] < -3).sum():,}  "
      f"({(df['HW5'] < -3).mean()*100:.1f}%)")
print(f"Stunted (HAZ < -2)   : {(df['HW70'] < -2).sum():,}  "
      f"({(df['HW70'] < -2).mean()*100:.1f}%)")
print(f"Anaemic (HW57 >= 1)  : {(df['HW57'] >= 1).sum():,}  "
      f"({(df['HW57'] >= 1).mean()*100:.1f}%)")
print()
print(f"Male / Female        : "
      f"{(df['sex']==0).sum():,} / {(df['sex']==1).sum():,}")
print(f"Urban / Rural        : "
      f"{(df['V025']==1).sum():,} / {(df['V025']==2).sum():,}")
print(f"Wealth dist          : "
      f"{df['V190'].value_counts().sort_index().to_dict()}")
print(f"Mother edu dist      : "
      f"{df['V106'].value_counts().sort_index().to_dict()}")
print(f"States covered       : {df['V024'].nunique()}")
print(f"Food cols            : {len(food_cols)}")
print("=" * 55)

sex  : {0: 120665, 1: 112255}
WHZ  : mean=-1.21  missing=36,050
HAZ  : mean=-1.31  missing=26,895
HW3  : min=2.0  max=14.0  missing=21,756
HW56 : min=3.0  max=20.0  missing=49,301
HW57 : {0.0: 4050, 1.0: 66709, 2.0: 52703, 3.0: 60393}
HW6 re-derived from HW5
HW6  (wasted)  : 0=173322  1=59598
HW7  (SAM)     : 0=2169  1=2154
HW71 (stunted) : 0=321  1=380
HW72 (underweight) : 0=540  1=556

Total remaining NaN : 0

FINAL DATASET QUALITY REPORT
Total rows           : 232,920
Total columns        : 41
Remaining NaN        : 0

WHZ  (HW5)           : -6.00 to 6.00  mean=-1.22
HAZ  (HW70)          : -6.00 to 6.00  mean=-1.33
Weight kg  (HW3)     : 2.0 to 14.0
Haemoglobin (HW56)   : 3.0 to 20.0

Wasted  (WHZ < -2)   : 59,598  (25.6%)
SAM     (WHZ < -3)   : 22,953  (9.9%)
Stunted (HAZ < -2)   : 73,072  (31.4%)
Anaemic (HW57 >= 1)  : 228,870  (98.3%)

Male / Female        : 120,665 / 112,255
Urban / Rural        : 47,199 / 185,721
Wealth dist          : {1.0: 63406, 2.0: 54463, 3.0: 45083, 4.0: 

In [24]:
import numpy as np
import pandas as pd

def create_target_labels(df):
    """
    Create 3 target variables:
    1. malnutrition_type
    2. deficiency_type
    3. severity_score (0-100)
    """
    df = df.copy()

    # Required columns for this logic
    required = ['HW5', 'HW70', 'HW56', 'HW57', 'H22']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    # Ensure numeric
    for col in ['HW5', 'HW70', 'HW56', 'HW57', 'H22', 'HW4', 'HW72']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # ── TARGET 1: Malnutrition type ─────────────────────────────────────
    conditions = [
        (df['HW5'] < -3),                               # SAM
        (df['HW5'] >= -3) & (df['HW5'] < -2),          # MAM
        (df['HW5'] >= -2) & (df['HW70'] < -2),         # Stunted only
        (df['HW5'] >= -2) & (df['HW70'] >= -2),        # Normal
    ]
    choices = ['SAM', 'MAM', 'Stunted', 'Normal']
    df['malnutrition_type'] = np.select(conditions, choices, default='Unknown')

    # Handle HW57 whether encoded as 0-3 (cleaned) or 1-4 (raw-style)
    hw57_max = df['HW57'].dropna().max() if not df['HW57'].dropna().empty else np.nan
    if pd.notna(hw57_max) and hw57_max > 3:
        # 1-4 coding
        anaemia_modsev = df['HW57'].isin([3, 4])
        anaemia_score_map = {1: 0, 2: 10, 3: 20, 4: 30}
    else:
        # 0-3 coding
        anaemia_modsev = df['HW57'].isin([2, 3])
        anaemia_score_map = {0: 0, 1: 10, 2: 20, 3: 30}

    # ── TARGET 2: Deficiency type ───────────────────────────────────────
    df['iron_risk'] = ((anaemia_modsev) | (df['HW56'] < 10.0)).astype('int8')
    df['vita_risk'] = (((df['H22'] == 0) & (df['HW70'] < -2))).astype('int8')

    # PEM proxy: use HW4 if present, else fallback to HW72 flag
    if 'HW4' in df.columns:
        pem_underweight = (df['HW4'] < -3)
    elif 'HW72' in df.columns:
        pem_underweight = (df['HW72'] == 1)
    else:
        pem_underweight = pd.Series(False, index=df.index)

    df['pem_risk'] = (((df['HW5'] < -2) | pem_underweight)).astype('int8')

    df['deficiency_type'] = np.select(
        [df['pem_risk'] == 1, df['iron_risk'] == 1, df['vita_risk'] == 1],
        ['Protein_Energy', 'Iron_Anaemia', 'Vitamin_A'],
        default='None'
    )

    # ── TARGET 3: Severity score (0-100) ────────────────────────────────
    whz_component = (-df['HW5']).clip(lower=0, upper=4) * 10          # max 40
    haz_component = (-df['HW70']).clip(lower=0, upper=(30/7)) * 7     # max 30
    anaemia_component = df['HW57'].map(anaemia_score_map).fillna(0)    # max 30

    df['severity_score'] = (whz_component + haz_component + anaemia_component).clip(0, 100).round(1)

    return df


# Run
df = create_target_labels(df)

# Check distribution
print("\nMalnutrition Type Distribution:")
print(df['malnutrition_type'].value_counts(dropna=False))

print("\nDeficiency Type Distribution:")
print(df['deficiency_type'].value_counts(dropna=False))

print("\nSeverity Score Distribution:")
print(df['severity_score'].describe())


Malnutrition Type Distribution:
malnutrition_type
Normal     159107
MAM         36645
SAM         22953
Stunted     14215
Name: count, dtype: int64

Deficiency Type Distribution:
deficiency_type
Protein_Energy    232543
Iron_Anaemia         331
None                  43
Vitamin_A              3
Name: count, dtype: int64

Severity Score Distribution:
count   232920.000
mean        42.020
std         20.613
min          0.000
25%         30.000
50%         37.800
75%         55.800
max        100.000
Name: severity_score, dtype: float64


In [25]:
import numpy as np
import pandas as pd

# ---------------------------
# 1) CHECK why deficiency_type is collapsed
# ---------------------------
print("=== Column availability ===")
for c in ['HW5','HW70','HW56','HW57','H22','HW4','HW72','deficiency_type']:
    print(f"{c:15} -> {'YES' if c in df.columns else 'NO'}")

print("\n=== Basic ranges / uniques ===")
if 'HW5' in df.columns:
    print("HW5  min/max:", pd.to_numeric(df['HW5'], errors='coerce').min(), pd.to_numeric(df['HW5'], errors='coerce').max())
if 'HW70' in df.columns:
    print("HW70 min/max:", pd.to_numeric(df['HW70'], errors='coerce').min(), pd.to_numeric(df['HW70'], errors='coerce').max())
if 'HW56' in df.columns:
    s = pd.to_numeric(df['HW56'], errors='coerce')
    print("HW56 min/max:", s.min(), s.max())
if 'HW57' in df.columns:
    print("HW57 value_counts:\n", pd.to_numeric(df['HW57'], errors='coerce').value_counts(dropna=False).sort_index().head(10))
if 'H22' in df.columns:
    print("H22 value_counts:\n", pd.to_numeric(df['H22'], errors='coerce').value_counts(dropna=False).sort_index().head(10))
if 'HW4' in df.columns:
    s = pd.to_numeric(df['HW4'], errors='coerce')
    print("HW4 min/max:", s.min(), s.max(), "| NaN:", s.isna().sum())

# ---------------------------
# 2) REBUILD deficiency_type robustly
#    (no hard dependency on HW4)
# ---------------------------
def rebuild_deficiency_type(df):
    d = df.copy()

    for col in ['HW5','HW70','HW56','HW57','H22','HW4','HW72']:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors='coerce')

    # Detect HW57 coding (0-3 cleaned vs 1-4 raw)
    hw57_non_na = d['HW57'].dropna()
    if hw57_non_na.empty:
        anaemia_modsev = pd.Series(False, index=d.index)
        anaemia_score = pd.Series(0, index=d.index)
    else:
        if hw57_non_na.max() > 3:   # 1-4 coding
            anaemia_modsev = d['HW57'].isin([3, 4])
            anaemia_score = d['HW57'].map({1:0, 2:10, 3:20, 4:30}).fillna(0)
        else:                       # 0-3 coding
            anaemia_modsev = d['HW57'].isin([2, 3])
            anaemia_score = d['HW57'].map({0:0, 1:10, 2:20, 3:30}).fillna(0)

    d['iron_risk'] = (anaemia_modsev | (d['HW56'] < 10.0)).astype(int)
    d['vita_risk'] = ((d['H22'] == 0) & (d['HW70'] < -2)).astype(int)

    # PEM proxy: WHZ wasting + underweight severe (if HW4 available and valid), else fallback HW72
    if 'HW4' in d.columns:
        hw4 = d['HW4'].copy()
        # If HW4 looks like scaled integers (e.g., -600..600), convert
        if hw4.dropna().abs().median() > 10:
            hw4 = hw4 / 100.0
        underweight_severe = (hw4 < -3)
    elif 'HW72' in d.columns:
        underweight_severe = (d['HW72'] == 1)
    else:
        underweight_severe = pd.Series(False, index=d.index)

    d['pem_risk'] = ((d['HW5'] < -2) | underweight_severe).astype(int)

    d['deficiency_type'] = np.select(
        [d['pem_risk'] == 1, d['iron_risk'] == 1, d['vita_risk'] == 1],
        ['Protein_Energy', 'Iron_Anaemia', 'Vitamin_A'],
        default='None'
    )

    return d

df = rebuild_deficiency_type(df)

print("\n=== Rebuilt deficiency_type distribution ===")
print(df['deficiency_type'].value_counts(dropna=False))

print("\n=== Component prevalence (%) ===")
for c in ['pem_risk','iron_risk','vita_risk']:
    print(f"{c:10}: {df[c].mean()*100:.2f}%")

=== Column availability ===
HW5             -> YES
HW70            -> YES
HW56            -> YES
HW57            -> YES
H22             -> YES
HW4             -> NO
HW72            -> YES
deficiency_type -> YES

=== Basic ranges / uniques ===
HW5  min/max: -6.0 6.0
HW70 min/max: -6.0 6.0
HW56 min/max: 3.0 20.0
HW57 value_counts:
 HW57
0.000      4050
1.000    115774
2.000     52703
3.000     60393
Name: count, dtype: int64
H22 value_counts:
 H22
0.000    205220
1.000     27700
Name: count, dtype: int64

=== Rebuilt deficiency_type distribution ===
deficiency_type
Protein_Energy    232543
Iron_Anaemia         331
None                  43
Vitamin_A              3
Name: count, dtype: int64

=== Component prevalence (%) ===
pem_risk  : 99.84%
iron_risk : 78.84%
vita_risk : 27.35%


In [26]:
import numpy as np
import pandas as pd

# Recompute clean deficiency targets
for c in ['HW5', 'HW56', 'HW57', 'HW70', 'H22']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# HW57 is already 0-3 in your data:
# 0 none, 1 mild, 2 moderate, 3 severe
df['iron_risk'] = ((df['HW57'] >= 2) | (df['HW56'] < 10.0)).astype(int)

df['vita_risk'] = (
    (df['H22'] == 0) &      # no Vit A supplement
    (df['HW70'] < -2)       # stunted
).astype(int)

# IMPORTANT FIX: strict PEM proxy from WHZ only (since HW4 missing)
df['pem_risk'] = (df['HW5'] < -3).astype(int)   # SAM only

# Priority label
df['deficiency_type'] = np.select(
    [df['pem_risk'] == 1, df['iron_risk'] == 1, df['vita_risk'] == 1],
    ['Protein_Energy', 'Iron_Anaemia', 'Vitamin_A'],
    default='None'
)

print("\nDeficiency Type Distribution (fixed):")
print(df['deficiency_type'].value_counts(dropna=False))

print("\nComponent prevalence (%):")
for c in ['pem_risk', 'iron_risk', 'vita_risk']:
    print(f"{c:10}: {df[c].mean()*100:.2f}%")


Deficiency Type Distribution (fixed):
deficiency_type
Iron_Anaemia      162140
None               42771
Protein_Energy     22953
Vitamin_A           5056
Name: count, dtype: int64

Component prevalence (%):
pem_risk  : 9.85%
iron_risk : 78.84%
vita_risk : 27.35%


In [27]:
import numpy as np
import pandas as pd

def build_feature_matrix(df):
    """Build robust feature matrix from cleaned DHS-style dataframe."""
    X = pd.DataFrame(index=df.index)

    def first_available(cols, default=np.nan):
        for c in cols:
            if c in df.columns:
                return df[c]
        return pd.Series(default, index=df.index)

    # ── Demographic ───────────────────────────────────────────────────────
    X['age_months'] = first_available(['HW1'])

    # Prefer cleaned sex if present; else map HW2
    if 'sex' in df.columns:
        X['sex'] = pd.to_numeric(df['sex'], errors='coerce')
    else:
        hw2 = pd.to_numeric(first_available(['HW2']), errors='coerce')
        # supports both 1/2 coding and 0/1 coding
        X['sex'] = hw2.map({1: 0, 2: 1}).fillna(hw2.where(hw2.isin([0, 1])))

    v025 = pd.to_numeric(first_available(['V025']), errors='coerce')
    X['rural'] = v025.map({1: 0, 2: 1})

    # ── Anthropometric ────────────────────────────────────────────────────
    X['weight_kg'] = pd.to_numeric(first_available(['HW3']), errors='coerce')
    X['height_cm'] = pd.to_numeric(first_available(['HW8']), errors='coerce')   # may be absent
    X['waz'] = pd.to_numeric(first_available(['HW4']), errors='coerce')          # may be absent
    X['whz'] = pd.to_numeric(first_available(['HW5']), errors='coerce')
    X['haz'] = pd.to_numeric(first_available(['HW70']), errors='coerce')
    X['haemoglobin'] = pd.to_numeric(first_available(['HW56']), errors='coerce')

    # ── Socioeconomic ─────────────────────────────────────────────────────
    X['wealth_index'] = pd.to_numeric(first_available(['V190', 'HC63']), errors='coerce')
    X['mother_edu'] = pd.to_numeric(first_available(['V106', 'HC61']), errors='coerce')

    # Cooking fuel (prefer V161; fallback V469A if you used that earlier)
    fuel = pd.to_numeric(first_available(['V161', 'V469A']), errors='coerce')
    X['clean_fuel'] = fuel.isin([1, 2, 3]).astype(int)

    water = pd.to_numeric(first_available(['V113']), errors='coerce')
    X['improved_water'] = water.isin([11, 12, 13, 14, 21, 31, 41, 51, 71, 72]).astype(int)

    sani = pd.to_numeric(first_available(['V116']), errors='coerce')
    X['improved_sanitation'] = sani.isin([11, 12, 13, 14, 15, 21, 22, 41]).astype(int)

    # ── Health history ────────────────────────────────────────────────────
    X['fever_2wks'] = pd.to_numeric(first_available(['H11']), errors='coerce').fillna(0)
    X['vit_a_suppl'] = pd.to_numeric(first_available(['H22']), errors='coerce').fillna(0)
    X['deworming'] = pd.to_numeric(first_available(['H43']), errors='coerce').fillna(0)

    # ── Dietary recall ────────────────────────────────────────────────────
    food_map = {
        'V414A': 'food_grain', 'V414B': 'food_legume', 'V414C': 'food_dairy',
        'V414D': 'food_flesh', 'V414E': 'food_egg', 'V414F': 'food_vita_veg',
        'V414G': 'food_other_veg', 'V414H': 'food_vita_fruit', 'V414I': 'food_other_fruit',
        'V414J': 'food_organ', 'V414K': 'food_processed', 'V414L': 'food_sweet_drink',
        'V414M': 'food_nuts', 'V414N': 'food_breastmilk', 'V414O': 'food_formula',
        'V414P': 'food_thin_porridge', 'V414Q': 'food_thick_porridge',
        'V414R': 'food_fortified', 'V414S': 'food_other'
    }
    for c, name in food_map.items():
        X[name] = pd.to_numeric(first_available([c]), errors='coerce').fillna(0).astype(int)

    food_groups = [
        'food_grain', 'food_legume', 'food_dairy', 'food_flesh',
        'food_egg', 'food_vita_veg', 'food_other_veg', 'food_vita_fruit'
    ]
    X['dietary_diversity'] = X[food_groups].sum(axis=1)

    X['anaemia_level'] = pd.to_numeric(first_available(['HW57']), errors='coerce').fillna(0)

    # Final numeric cleanup
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')
        if X[c].isna().any():
            X[c] = X[c].fillna(X[c].median())

    return X


X = build_feature_matrix(df)
y_class = df['deficiency_type']
y_maln  = df['malnutrition_type']
y_score = df['severity_score']

print(f"\nFeature matrix shape: {X.shape}")
print(f"\nFeature columns:\n{X.columns.tolist()}")
print(f"\nClass distribution:\n{y_class.value_counts(dropna=False)}")


Feature matrix shape: (232920, 38)

Feature columns:
['age_months', 'sex', 'rural', 'weight_kg', 'height_cm', 'waz', 'whz', 'haz', 'haemoglobin', 'wealth_index', 'mother_edu', 'clean_fuel', 'improved_water', 'improved_sanitation', 'fever_2wks', 'vit_a_suppl', 'deworming', 'food_grain', 'food_legume', 'food_dairy', 'food_flesh', 'food_egg', 'food_vita_veg', 'food_other_veg', 'food_vita_fruit', 'food_other_fruit', 'food_organ', 'food_processed', 'food_sweet_drink', 'food_nuts', 'food_breastmilk', 'food_formula', 'food_thin_porridge', 'food_thick_porridge', 'food_fortified', 'food_other', 'dietary_diversity', 'anaemia_level']

Class distribution:
deficiency_type
Iron_Anaemia      162140
None               42771
Protein_Energy     22953
Vitamin_A           5056
Name: count, dtype: int64


In [29]:
from sklearn.impute import SimpleImputer

anthropometric = ['weight_kg', 'height_cm', 'waz', 'whz', 'haz', 'haemoglobin', 'anaemia_level']
categorical = ['sex', 'rural', 'wealth_index', 'mother_edu', 'clean_fuel',
               'improved_water', 'improved_sanitation', 'fever_2wks',
               'vit_a_suppl', 'deworming']

anthropometric = [c for c in anthropometric if c in X_train.columns]
categorical = [c for c in categorical if c in X_train.columns]

# columns that are fully missing in TRAIN
all_na_anthro = [c for c in anthropometric if X_train[c].isna().all()]
anthro_ok = [c for c in anthropometric if c not in all_na_anthro]

print("All-NaN anthropometric columns (dropping):", all_na_anthro)

# Drop all-NaN columns from all splits
if all_na_anthro:
    X_train = X_train.drop(columns=all_na_anthro)
    X_val   = X_val.drop(columns=all_na_anthro)
    X_test  = X_test.drop(columns=all_na_anthro)

# Fit on train only
cat_imputer = SimpleImputer(strategy='most_frequent')
anthro_imputer = SimpleImputer(strategy='median')

if categorical:
    cat_imputer.fit(X_train[categorical])

if anthro_ok:
    anthro_imputer.fit(X_train[anthro_ok])

def transform_split(df_split):
    out = df_split.copy()
    if categorical:
        out[categorical] = cat_imputer.transform(out[categorical])
    if anthro_ok:
        out[anthro_ok] = anthro_imputer.transform(out[anthro_ok])
    return out

X_train = transform_split(X_train)
X_val   = transform_split(X_val)
X_test  = transform_split(X_test)

print("Missing after imputation:",
      X_train.isna().sum().sum(),
      X_val.isna().sum().sum(),
      X_test.isna().sum().sum())
print("Final feature count:", X_train.shape[1])

All-NaN anthropometric columns (dropping): ['height_cm', 'waz']
Missing after imputation: 0 0 0
Final feature count: 36


In [30]:
import os
import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# ---------------------------------------------------
# 1) Keep usable rows
# ---------------------------------------------------
mask = y_class.notna() & (y_class != 'Unknown')
X_clean = X.loc[mask].copy()
y_clean = y_class.loc[mask].copy()
y_score_clean = y_score.loc[mask].copy()

print(f"Usable rows: {len(X_clean)}")
print(f"\nClass distribution:\n{y_clean.value_counts(dropna=False)}")
print(f"\nClass balance (%):\n{(y_clean.value_counts(normalize=True)*100).round(2)}")

# ---------------------------------------------------
# 2) Split first (important), then fit imputers on TRAIN only
# ---------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

# aligned regression targets
y_score_train = y_score_clean.loc[X_train.index]
y_score_val   = y_score_clean.loc[X_val.index]
y_score_test  = y_score_clean.loc[X_test.index]

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# ---------------------------------------------------
# 3) Imputation config
# ---------------------------------------------------
anthropometric = ['weight_kg', 'height_cm', 'waz', 'whz', 'haz', 'haemoglobin', 'anaemia_level']
categorical = ['sex', 'rural', 'wealth_index', 'mother_edu', 'clean_fuel',
               'improved_water', 'improved_sanitation', 'fever_2wks',
               'vit_a_suppl', 'deworming']

# keep only columns that exist
anthropometric = [c for c in anthropometric if c in X_train.columns]
categorical = [c for c in categorical if c in X_train.columns]

print("\nMissing % in TRAIN before imputation:")
missing_pct = (X_train.isnull().sum() / len(X_train) * 100).round(2)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

cat_imputer = SimpleImputer(strategy='most_frequent')
anthro_imputer = SimpleImputer(strategy='median')

def transform_split(df_split):
    df_out = df_split.copy()
    if categorical:
        df_out[categorical] = cat_imputer.transform(df_out[categorical])
    if anthropometric:
        df_out[anthropometric] = anthro_imputer.transform(df_out[anthropometric])
    return df_out

# fit on train only
if categorical:
    cat_imputer.fit(X_train[categorical])
if anthropometric:
    anthro_imputer.fit(X_train[anthropometric])

# transform all splits
X_train = transform_split(X_train)
X_val   = transform_split(X_val)
X_test  = transform_split(X_test)

print(f"\nMissing after imputation (train/val/test): "
      f"{X_train.isnull().sum().sum()} / {X_val.isnull().sum().sum()} / {X_test.isnull().sum().sum()}")

# ---------------------------------------------------
# 4) Save outputs
# ---------------------------------------------------
os.makedirs('processed', exist_ok=True)

X_train.to_csv(r'processed\X_train.csv', index=False)
X_val.to_csv(r'processed\X_val.csv', index=False)
X_test.to_csv(r'processed\X_test.csv', index=False)

y_train.rename('deficiency_type').to_csv(r'processed\y_train.csv', index=False)
y_val.rename('deficiency_type').to_csv(r'processed\y_val.csv', index=False)
y_test.rename('deficiency_type').to_csv(r'processed\y_test.csv', index=False)

y_score_train.rename('severity_score').to_csv(r'processed\y_score_train.csv', index=False)
y_score_val.rename('severity_score').to_csv(r'processed\y_score_val.csv', index=False)
y_score_test.rename('severity_score').to_csv(r'processed\y_score_test.csv', index=False)

joblib.dump(cat_imputer, r'processed\cat_imputer.pkl')
joblib.dump(anthro_imputer, r'processed\anthro_imputer.pkl')
joblib.dump(list(X_train.columns), r'processed\feature_columns.pkl')

print("\nAll files saved to processed\\")
print("Processing complete. Ready for model training.")

Usable rows: 232920

Class distribution:
deficiency_type
Iron_Anaemia      162140
None               42771
Protein_Energy     22953
Vitamin_A           5056
Name: count, dtype: int64

Class balance (%):
deficiency_type
Iron_Anaemia     69.610
None             18.360
Protein_Energy    9.850
Vitamin_A         2.170
Name: proportion, dtype: float64

Train: 167702 | Val: 18634 | Test: 46584

Missing % in TRAIN before imputation:
height_cm   100.000
waz         100.000
dtype: float64


ValueError: Columns must be same length as key

In [31]:
import os, joblib

os.makedirs('processed', exist_ok=True)

X_train.to_csv(r'processed\X_train.csv', index=False)
X_val.to_csv(r'processed\X_val.csv', index=False)
X_test.to_csv(r'processed\X_test.csv', index=False)

y_train.rename('deficiency_type').to_csv(r'processed\y_train.csv', index=False)
y_val.rename('deficiency_type').to_csv(r'processed\y_val.csv', index=False)
y_test.rename('deficiency_type').to_csv(r'processed\y_test.csv', index=False)

y_score_train.rename('severity_score').to_csv(r'processed\y_score_train.csv', index=False)
y_score_val.rename('severity_score').to_csv(r'processed\y_score_val.csv', index=False)
y_score_test.rename('severity_score').to_csv(r'processed\y_score_test.csv', index=False)

joblib.dump(cat_imputer, r'processed\cat_imputer.pkl')
joblib.dump(anthro_imputer, r'processed\anthro_imputer.pkl')
joblib.dump(list(X_train.columns), r'processed\feature_columns.pkl')

print("All files saved to processed\\")
print("Processing complete. Ready for model training.")

All files saved to processed\
Processing complete. Ready for model training.
